# Anime Recommender Inference Test 
Here we can test creating user representations from arbitrary anime ID lists and prompts.

In [1]:
#%cd hybrid-recsys/
%cd ..
!pip install -e .

/home/invogue/Documents/Projects/hybrid-recsys
Obtaining file:///home/invogue/Documents/Projects/hybrid-recsys
  Preparing metadata (setup.py) ... done
  Attempting uninstall: hybrid_recsys
    Found existing installation: hybrid_recsys 0.1
    Uninstalling hybrid_recsys-0.1:
      Successfully uninstalled hybrid_recsys-0.1
  Running setup.py develop for hybrid_recsys


In [2]:
import pickle
import torch
import torch.nn.functional as F
import numpy as np
from attentionrec.models.attentionrec import TransformerRecommendationModel
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = os.getcwd()  # current working dir
model_path = os.path.join(PROJECT_ROOT, "checkpoints/attentionrec/latest_checkpoint.pt")
embeddings_path = os.path.join(PROJECT_ROOT, "data/processed/attentionrec/anime-embeddings.pkl")

# %%
# Load embeddings
with open(embeddings_path, "rb") as f:
    emb_data = pickle.load(f)
    
description_embeddings = torch.tensor(emb_data['embeddings'], device=device)
embedding_dim = emb_data['embedding_dim']
anime_id_to_idx = emb_data['anime_id_to_idx']
idx_to_anime_id = {v: k for k, v in anime_id_to_idx.items()}

# %%
# Load model
model = TransformerRecommendationModel(
    embedding_dim=embedding_dim,
    num_heads=4,
    num_layers=2,
    dropout_rate=0.1,
).to(device)

# Explicitly allow unsafe globals (full unpickling)
checkpoint = torch.load(model_path, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Model loaded successfully.")

# Function to create a "user" embedding from a list of anime IDs
def create_user_embedding(anime_id_list):
    """
    anime_id_list: list of anime IDs to build the user representation from
    B: number of items to sample if len(list) > B
    """
    # Filter out anime IDs not in embeddings
    valid_ids = [i for i in anime_id_list if i in anime_id_to_idx]
    if not valid_ids:
        raise ValueError("No valid anime IDs found in embeddings!")

    indices = [anime_id_to_idx[i] for i in valid_ids]
    user_embeddings = description_embeddings[indices]
    print(f"User embedding shape before model: {user_embeddings.shape}")
    user_rep = model(user_embeddings.unsqueeze(0)).squeeze()
    return user_rep

# Function to compute top-K recommendations given a user embedding
def recommend_topk(user_rep, K=10, mask_ids=None):
    """
    user_rep: tensor [embedding_dim]
    K: number of recommendations
    mask_ids: optional list of anime IDs to mask (e.g., already seen)
    """
    scores = user_rep @ description_embeddings.T  # [num_items]

    # Mask any IDs
    if mask_ids:
        mask_idx = [anime_id_to_idx[i] for i in mask_ids if i in anime_id_to_idx]
        scores[mask_idx] = -float("inf")

    topk_idx = torch.topk(scores, K).indices.cpu().numpy()
    topk_anime_ids = [idx_to_anime_id[i] for i in topk_idx]
    return topk_anime_ids

def create_augmented_user_embedding(anime_id_list, prompt_embedding=None, prompt_weight=4.0, max_items=None):
    valid_ids = [i for i in anime_id_list if i in anime_id_to_idx]
    if not valid_ids:
        raise ValueError("No valid anime IDs found in embeddings!")

    if max_items is not None and len(valid_ids) > max_items:
        valid_ids = valid_ids[:max_items]  # prevent long sequences

    indices = [anime_id_to_idx[i] for i in valid_ids]
    user_item_embeddings = description_embeddings[indices]  # [num_items, D9
    if user_item_embeddings.dim() == 2:
        user_item_embeddings = user_item_embeddings.unsqueeze(0)  # (1, num_items, D)

    if prompt_embedding is not None:
        if prompt_embedding.dim() == 1:
            prompt_embedding = prompt_embedding.unsqueeze(0)  # (1, D)
        elif prompt_embedding.dim() > 2:
            raise ValueError(f"Prompt embedding has too many dimensions: {prompt_embedding.shape}")

        prompt_embedding_scaled = prompt_embedding.unsqueeze(1) * prompt_weight

        combined_embeddings = torch.cat([user_item_embeddings, prompt_embedding_scaled], dim=1)  # (1, num_items+1, D)
    else:
        combined_embeddings = user_item_embeddings  # (1, num_items, D)
    assert combined_embeddings.dim() == 3, f"Combined embeddings must be 3D, got {combined_embeddings.shape}"

    with torch.no_grad():
        user_rep = model(combined_embeddings).squeeze(0)  # remove batch dim

    return user_rep

Model loaded successfully.


In [3]:
#load anime csv to get the title
import pandas as pd
csv_path = "data/raw/mal/anime-dataset-2023.csv"

# Read CSV (assuming columns include 'anime_id' and 'title')
df = pd.read_csv(csv_path)

# Create a mapping from ID to title
id2title = dict(zip(df['anime_id'], df['Name']))

# Synthetic users
create some sample users to see how the model performs on 

In [4]:
archetype_users = {
    "shounen_watcher": [20, 21, 5114, 813],
    "slice_of_life_fan": [1735, 5680, 23273, 23287],
    "psych_fan": [1535, 9253, 13381, 9257],
    "game_strategist": [19815, 31240, 11757, 11399],
    "classic_veteran": [1, 97, 30, 2],
}

for u, anime_ids_for_user in archetype_users.items():
    print(u)
    
    # Create user embedding
    user_rep = create_user_embedding(anime_ids_for_user)
    
    # Get top-K recommendations
    top10 = recommend_topk(user_rep, K=10, mask_ids=anime_ids_for_user)
    print("Top-10 recommendations for synthetic user:", top10)
    
    # Get titles for your list of IDs
    titles = [id2title.get(aid, f"Unknown ID {aid}") for aid in top10]
    print(titles)

shounen_watcher
User embedding shape before model: torch.Size([4, 1024])
Top-10 recommendations for synthetic user: [121, 1535, 269, 225, 791, 39417, 32379, 223, 1292, 55453]
['Fullmetal Alchemist', 'Death Note', 'Bleach', 'Dragon Ball GT', 'Arion', 'Granbelm', 'Berserk', 'Dragon Ball', 'Afro Samurai', 'Naruto (2023)']
slice_of_life_fan
User embedding shape before model: torch.Size([3, 1024])
Top-10 recommendations for synthetic user: [1535, 6547, 46569, 19429, 50203, 50060, 27899, 37799, 29758, 7791]
['Death Note', 'Angel Beats!', 'Jigokuraku', 'Akuma no Riddle', 'Love Live! Superstar!! 2nd Season', 'Shadowverse Flame', 'Tokyo Ghoul √A', 'Tokyo Ghoul:re 2nd Season', 'Taboo Tattoo', 'K-On!!']
psych_fan
User embedding shape before model: torch.Size([2, 1024])
Top-10 recommendations for synthetic user: [31580, 28791, 6547, 36882, 38668, 49956, 384, 44200, 16890, 53012]
['Ajin', 'Gunslinger Stratos The Animation', 'Angel Beats!', 'Arifureta Shokugyou de Sekai Saikyou', 'Dorohedoro', 'Chan

# Augmented user embedding with a prompt
Because we used SentenceTransformer embeddings to train,
we can in theory use natural language prompts to generate user embeddings without needing to specify anime IDs. 
This is a powerful feature of using text-based embeddings and attention mechanisms, as it allows us to capture user preferences in a more flexible way.

We can see that the model starts recommending more "magical girl" anime like madoka magica after augmenting the user recommendation with a prompt (very cool) even though the actual recomendations arent the best (expected).

Might also just be recommend more magical anime.

We would probably get quite a lot better results if we also trained the embedding model (or just trained more epochs on our recommender).

In [5]:
from sentence_transformers import SentenceTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "stsb-roberta-large" # We need to use the same model as the one used to generate the description embeddings, otherwise the embeddings won't be compatible with the model's attention layers.


# Load model
print(f"Loading model {model_name}...")
embedding_model = SentenceTransformer(model_name)
embedding_model = embedding_model.to(device)

# Because we used SentenceTransformer embeddings to train,
# we can in theory use natural language prompts to generate user embeddings without needing to specify anime IDs. 
# This is a powerful feature of using text-based embeddings and attention mechanisms, as it allows us to capture user preferences in a more flexible way.

# Prepare prompt
prompt = "Magical girl anime"  # Example prompt describing a user archetype. In practice, you could have more complex prompts or even multiple prompts per user.
prompt_embedding = embedding_model.encode(prompt, normalize_embeddings=True, show_progress_bar=True, convert_to_tensor=True)  # SentenceTransformer


archetype_users = {
    "shounen_watcher": [20, 21, 5114, 813],
    "slice_of_life_fan": [1735, 5680, 23273, 23287],
    "psych_fan": [1535, 9253, 13381, 9257],
    "game_strategist": [19815, 31240, 11757, 11399],
    "classic_veteran": [1, 97, 30, 2],
}

for u, anime_ids_for_user in archetype_users.items():
    print(u)

    # Create augmented user embedding
    user_rep = create_augmented_user_embedding(anime_ids_for_user, prompt_embedding=prompt_embedding)

    # Get top-K recommendations
    top10 = recommend_topk(user_rep, K=10, mask_ids=anime_ids_for_user)
    titles = [id2title.get(aid, f"Unknown ID {aid}") for aid in top10]
    
    print("Top-10 recommendations for synthetic user:", top10)
    print(titles)

/home/invogue/Documents/Projects/hybrid-recsys/src/attentionrec/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model stsb-roberta-large...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 13765.21it/s]
RobertaModel LOAD REPORT from: sentence-transformers/stsb-roberta-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00,  7.54it/s]

shounen_watcher
Top-10 recommendations for synthetic user: [1535, 121, 223, 269, 225, 55453, 39417, 791, 46569, 32379]
['Death Note', 'Fullmetal Alchemist', 'Dragon Ball', 'Bleach', 'Dragon Ball GT', 'Naruto (2023)', 'Granbelm', 'Arion', 'Jigokuraku', 'Berserk']
slice_of_life_fan
Top-10 recommendations for synthetic user: [1535, 19429, 7662, 6547, 269, 121, 50060, 20, 4983, 46569]
['Death Note', 'Akuma no Riddle', 'Shinrei Tantei Yakumo', 'Angel Beats!', 'Bleach', 'Fullmetal Alchemist', 'Shadowverse Flame', 'Naruto', 'Hells', 'Jigokuraku']
psych_fan
Top-10 recommendations for synthetic user: [6547, 36882, 36266, 38668, 31580, 14345, 3342, 53012, 6880, 384]
['Angel Beats!', 'Arifureta Shokugyou de Sekai Saikyou', 'Mahou Shoujo Site', 'Dorohedoro', 'Ajin', 'Btooom!', 'Mnemosyne: Mnemosyne no Musume-tachi', 'Human Bug Daigaku', 'Deadman Wonderland', 'Gantz']
game_strategist
Top-10 recommendations for synthetic user: [1535, 36266, 9756, 19429, 178, 3342, 795, 29758, 52211, 37984]
['Death N